# 06c · 반사 원리, 최댓값, 도달 시간 (Reflection Principle, Maximum and Hitting Times)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Ross 12e §10.2 (도달시간·최댓값·BM 의 도박꾼 파산) · Shreve II §3.6 (도달시간 분포), §3.7 (반사 원리, $T_m$ 의 분포, $(M_t,B_t)$ 결합분포) · Lawler 2e §8.2 (강마르코프·반사 원리), §8.3 (영점 집합) · G&S 4e §13.4 (first passage times) · Karlin & Taylor 2e §7.3 · Broadie–Glasserman–Kou (1997) Thm 1.1 (격자 편향 보정; 교재 아님) |
| 선수 노트북 | 06b (격자 위 BM, 이차 변동), 06a (정의·자기유사성), 00b (MC 오차·SE) |
| 예상 소요 | 90분 |
| 상태 | draft |

06a–06b 에서 BM 의 경로를 만들고 그 거칠음을 쟀다. 이 노트북은 경로의 **극값**을 묻는다: $[0,t]$ 에서 얼마나 높이 올라가는가($M_t$), 수준 $a$ 에 언제 처음 닿는가($T_a$)? 답은 놀랍도록 단순하다 — "장벽에 닿은 순간부터 경로를 뒤집어도 확률이 같다"는 **반사 원리**(reflection principle) 하나로 $P(M_t\ge a)=2P(B_t\ge a)$ 가 나오고, 여기서 $T_a$ 의 밀도, 무한 평균, 아크사인 법칙까지 줄줄이 따라온다. 시뮬레이션 쪽에는 함정이 있다: 컴퓨터는 격자점만 보므로 **격자 사이의 봉우리를 놓친다**. 그 편향이 $\sqrt{\Delta t}$ 로만 줄어드는 것을 직접 재고, "격자점 사이는 브라운 다리" 라는 사실로 편향을 0 으로 만드는 방법을 배운다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm
from scipy.integrate import quad
from scipy.special import zeta
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.brownian import brownian_paths, running_max, first_passage_index
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("06c")
setup_plots()
N_PATHS = scale(20_000)        # 격자/다리 보정 실험의 경로 수; FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_EXACT = scale(1_000_000)     # 정확 표본기 1/Z^2 의 표본 수
N_STEPS_LIST = [10, 100, 1000] if fast() else [10, 100, 1000, 10_000]
BETA = -zeta(0.5) / np.sqrt(2 * np.pi)   # Broadie–Glasserman–Kou 상수 -zeta(1/2)/sqrt(2 pi) = 0.5826
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_EXACT={N_EXACT}, n_steps={N_STEPS_LIST}, beta={BETA:.6f}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (06b) 등간격 $n$ 분할의 이차 변동 $Q_n(T)$ 의 평균과 분산은?

<details><summary>정답</summary>

$E[Q_n(T)]=T$, $\mathrm{Var}(Q_n(T))=2T^2/n$. 고정 $n$ 에서는 폭 $T\sqrt{2/n}$ 의 확률변수이고, "$=T$" 는 $n\to\infty$ 극한이다.

</details>

**Q2.** (06b) 총변동이 무한인 이유를 한 줄로.

<details><summary>정답</summary>

$Q_n\le\max_i|\Delta_i|\cdot V_n$ 에서 좌변 $\to T>0$, $\max_i|\Delta_i|\to0$ (경로 연속성) 이므로 $V_n\to\infty$.

</details>

**Q3.** (06b) $\min(t_i,t_j)$ 의 Cholesky 인자는 무엇인가?

<details><summary>정답</summary>

$\sqrt{\Delta t}$ 로 채운 하삼각행렬 — 곧 누적합 연산자. 그래서 "증분을 더하라" 와 "공분산 $\min$ 을 Cholesky 로 풀어라" 가 같은 경로를 준다.

</details>

**Q4.** (06a) 자기유사성(self-similarity)을 정확히 진술하라.

<details><summary>정답</summary>

임의의 $c>0$ 에 대해 $(c^{-1/2}B_{ct})_{t\ge0}$ 도 표준 BM 이다. 오늘은 이것으로 $T_a\overset d=a^2T_1$ 을 얻는다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **러닝 최댓값**(running maximum) $M_t=\max_{s\le t}B_s$ 와 **도달시간**(hitting time, first passage time) $T_a=\inf\{t\ge0:B_t=a\}$ ($a>0$). 경로가 연속이므로 $a$ 를 뛰어넘을 수 없고, 따라서 두 사건이 같다: $\{T_a\le t\}=\{M_t\ge a\}$.
- **강마르코프 성질**(strong Markov property): 정지시간 $T$ ($T<\infty$ a.s.) 에 대해 $(B_{T+s}-B_T)_{s\ge0}$ 는 $\mathcal F_T$ 와 독립인 표준 BM 이다 (Lawler 2e §8.2, Shreve II §3.7.1 — 인용). $T_a$ 는 정지시간이다.
- **반사된 경로**(reflected path): $\tilde B_t=B_t$ ($t<T_a$), $\tilde B_t=2a-B_t$ ($t\ge T_a$). 강마르코프 성질과 증분의 대칭성으로 $\tilde B$ 도 표준 BM 이다 — 이것이 반사 원리의 내용이다.
- **격자 추정량** ($\Delta t=T/n$): $M^{(n)}_t=\max_{k\le t/\Delta t}B_{k\Delta t}\le M_t$ (항상 과소), $T^{(n)}_a=\min\{k\Delta t:B_{k\Delta t}\ge a\}\ge T_a$ (항상 과대). 06a 이후 우리가 그려 온 모든 "최댓값" 은 이것이었다.
- **브라운 다리 최댓값 보정**(Brownian-bridge correction): $B_{t_i}=x$, $B_{t_{i+1}}=y$ 가 주어졌을 때 구간 $[t_i,t_{i+1}]$ 안의 최댓값 $m$ 은
  $$P(m\le z\mid x,y)=1-\exp\!\Big(-\frac{2(z-x)(z-y)}{\Delta t}\Big),\qquad z\ge\max(x,y).$$
  역변환으로 $m=\tfrac12\big(x+y+\sqrt{(y-x)^2-2\Delta t\ln U}\big)$, $U\sim U(0,1)$. 구간별 다리는 (양 끝점이 주어지면) 서로 독립이므로 격자 시각 $k\Delta t$ 까지의 **정확한** 러닝 최댓값은 처음 $k$ 개 구간 최댓값의 max 다. 같은 원리로, $x,y$ 가 같은 부호일 때 구간 안에서 0 을 지날 확률은 $\exp(-2xy/\Delta t)$ 이다.
- **아크사인 법칙의 세 통계량** ($[0,1]$): 양수 체류 시간 비율 $A_1=\int_0^1\mathbf 1\{B_s>0\}\,ds$, 마지막 영점 $L_1=\sup\{s\le1:B_s=0\}$, 최댓값 위치 $\arg\max_{[0,1]}B$. 셋 다 $P(\cdot\le x)=\frac2\pi\arcsin\sqrt x$.

### 2.2 정리 1 — 반사 원리와 최댓값의 분포 (Shreve II §3.7, Ross 12e §10.2, G&S 4e §13.4)

$a>0$ 에 대해
$$P(M_t\ge a)=2P(B_t\ge a)=P(|B_t|\ge a)=2\Big(1-\Phi\big(a/\sqrt t\big)\Big).$$
결합분포: $b\le a$ 에 대해 $P(M_t\ge a,\ B_t\le b)=P(B_t\ge 2a-b)=1-\Phi\big((2a-b)/\sqrt t\big)$. 따라서 $M_t\overset d=|B_t|$, 밀도 $\frac{2}{\sqrt t}\phi(x/\sqrt t)$ ($x\ge0$), $E[M_t]=\sqrt{2t/\pi}$.

가정이 왜 필요한가:
- **강마르코프 성질** (정지시간 $T_a$ 이후가 새 BM): 반사된 경로가 또 BM 이 되는 근거. 드리프트가 있어도 강마르코프는 성립하지만 다음 항목이 깨진다.
- **대칭성** (증분 분포가 부호 반전에 불변): $T_a$ 이후 위/아래로 갈 확률이 같아야 $P(M_t\ge a,B_t<a)=P(M_t\ge a,B_t>a)$. 드리프트 BM 에서는 실패한다 (E3).
- **경로 연속성**: $B_{T_a}=a$ 정확히. 점프가 있으면 overshoot 이 생겨 반사 후 경로가 $a$ 에서 출발하지 않는다.

<details><summary>증명</summary>

연속성으로 $\{M_t\ge a\}=\{T_a\le t\}$. 강마르코프 성질에 의해 $T_a$ 이후의 $W_s=B_{T_a+s}-a$ 는 $\mathcal F_{T_a}$ 와 독립인 표준 BM 이고, 표준 BM 은 대칭($-W\overset d=W$)이므로 사건 $\{T_a\le t\}$ 위에서 $B_t-a=W_{t-T_a}$ 의 부호는 조건부로 공평하다:
$$P(T_a\le t,\ B_t<a)=P(T_a\le t,\ B_t>a)\qquad(B_t=a\text{ 는 확률 }0).$$
한편 $\{B_t>a\}\subset\{T_a\le t\}$ (연속 경로가 $a$ 위에 있으려면 먼저 $a$ 를 지나야 한다) 이므로 $P(T_a\le t,B_t>a)=P(B_t>a)$. 둘을 더하면
$$P(T_a\le t)=P(T_a\le t,B_t>a)+P(T_a\le t,B_t<a)=2P(B_t>a)=2\big(1-\Phi(a/\sqrt t)\big).$$
$B_t\sim N(0,t)$ 의 대칭성으로 $2P(B_t\ge a)=P(|B_t|\ge a)$.

**결합식.** 같은 반사로 $\{T_a\le t,\ B_t\le b\}$ 와 $\{T_a\le t,\ B_t\ge2a-b\}$ (수준 $a$ 를 기준으로 $b$ 의 거울상이 $2a-b$) 의 확률이 같다. $b\le a$ 이면 $2a-b\ge a$ 이므로 후자는 $\{B_t\ge2a-b\}$ 자체이고 (그 사건은 자동으로 $T_a\le t$ 를 함의), $P(B_t\ge2a-b)=1-\Phi((2a-b)/\sqrt t)$.

**$M_t\overset d=|B_t|$.** 모든 $a>0$ 에서 $P(M_t\ge a)=P(|B_t|\ge a)$ 이고 둘 다 $a\le0$ 에서 1 이므로 분포함수가 같다. $|B_t|=\sqrt t|Z|$ 의 밀도는 $\frac2{\sqrt t}\phi(x/\sqrt t)$, $E|Z|=\sqrt{2/\pi}$ 이므로 $E[M_t]=\sqrt{2t/\pi}$. $\square$

</details>

### 2.3 정리 2 — 도달시간 $T_a$ 의 분포 (Lévy 분포; Shreve II §3.6–3.7, Ross 12e §10.2)

$a>0$ 에 대해
$$P(T_a\le t)=2\Big(1-\Phi\big(a/\sqrt t\big)\Big),\qquad f_{T_a}(t)=\frac{a}{\sqrt{2\pi t^3}}\,e^{-a^2/2t}\quad(t>0).$$
$T_a\overset d=a^2/Z^2$ ($Z\sim N(0,1)$). $P(T_a<\infty)=1$ 이지만 $E[T_a]=\infty$. 중앙값은 $a^2/\Phi^{-1}(3/4)^2\approx2.198\,a^2$, 꼬리는 $P(T_a>t)\sim a\sqrt{2/(\pi t)}$. 자기유사성으로 $T_a\overset d=a^2T_1$.

가정이 왜 필요한가:
- 정리 1 의 가정 (대칭·연속·강마르코프) 이 전부다 — CDF 가 정리 1 그 자체이기 때문.
- $a>0$ 고정: $a\to0$ 이면 $T_a\to0$. BM 은 0 근처에서 즉시 양수가 되고(그리고 즉시 음수도 된다), 0 의 근방에 영점이 무한히 많다 (Lawler 2e §8.3).

<details><summary>증명</summary>

**CDF** 는 정리 1 과 $\{T_a\le t\}=\{M_t\ge a\}$ 에서. **밀도**는 미분: $\frac{d}{dt}\big[2(1-\Phi(a t^{-1/2}))\big]=-2\phi(a/\sqrt t)\cdot\big(-\tfrac a2t^{-3/2}\big)=\frac{a}{\sqrt{2\pi t^3}}e^{-a^2/2t}$.

**유한성.** $t\to\infty$ 에서 $a/\sqrt t\to0$, $\Phi\to\frac12$, CDF $\to1$: $P(T_a<\infty)=1$.

**무한 평균.** $E[T_a]=\int_0^\infty P(T_a>t)\,dt$ 이고 $P(T_a>t)=2\Phi(a/\sqrt t)-1$. $u=a/\sqrt t\to0$ 에서 $2\Phi(u)-1\approx2\phi(0)u=a\sqrt{2/\pi}\,t^{-1/2}$ 이므로 피적분함수가 $t^{-1/2}$ 처럼 감소하고 $\int^\infty t^{-1/2}dt=\infty$. (같은 계산이 절단 평균 $E[T_a\wedge c]\approx\sqrt{8c/\pi}\cdot a$ 의 성장을 준다.)

**$a^2/Z^2$ 표현.** $a^2/Z^2\le t\iff Z^2\ge a^2/t\iff|Z|\ge a/\sqrt t$ 의 확률은 $2(1-\Phi(a/\sqrt t))$ 로 위의 CDF 와 같다.

**중앙값.** $2(1-\Phi(a/\sqrt t))=\tfrac12\iff\Phi(a/\sqrt t)=\tfrac34\iff a/\sqrt t=\Phi^{-1}(3/4)=0.67449$, 즉 $t=a^2/0.67449^2=2.1981a^2$.

**자기유사성.** $B'_s=B_{a^2s}/a$ 가 표준 BM 이므로 $T_a=\inf\{t:B_t=a\}=a^2\inf\{s:B'_s=1\}\overset d=a^2T_1$. $\square$

</details>

### 2.4 정리 3 — 아크사인 법칙 (예고; Lawler 2e §8.3, G&S 4e §13.4, 인용)

$A_1$, $L_1$, $\arg\max_{[0,1]}B$ 는 모두 밀도 $\dfrac{1}{\pi\sqrt{x(1-x)}}$ (U 자형) 의 아크사인 분포를 따른다. 특히 $P(A_1\le\tfrac14)=P(L_1\le\tfrac14)=\frac2\pi\arcsin\tfrac12=\tfrac13$.

가정이 왜 필요한가:
- 자기유사성과 (강)마르코프 성질: 영점 이후 재출발하는 구조가 "마지막 영점" 의 분포를 결정한다.
- 이산 걸음에서는 $n\to\infty$ 극한으로만 성립한다 (01x 의 $n^{-1/2}$ 생존확률과 연결). 노트북에서는 시뮬레이션으로만 확인한다 (Mörters–Peres Thm 5.28 참고).

### 2.5 정리 4 — 격자 편향과 연속성 보정 (Broadie–Glasserman–Kou 1997 Thm 1.1, 인용)

등간격 격자 $\Delta t$ 의 최댓값에 대해
$$P\big(M^{(n)}_T\ge a\big)=P\big(M_T\ge a+\beta\sqrt{\Delta t}\big)+o(\sqrt{\Delta t}),\qquad\beta=-\frac{\zeta(1/2)}{\sqrt{2\pi}}=0.5826.$$
즉 격자는 장벽을 $\beta\sqrt{\Delta t}$ 만큼 바깥으로 민 것처럼 행동하고, $E[M_T]-E[M^{(n)}_T]\approx\beta\sqrt{\Delta t}$. 편향은 $\sqrt{\Delta t}$ 로만 줄어든다.

가정이 왜 필요한가:
- 등간격 격자·연속 경로: 랜덤 워크가 수준을 넘을 때의 overshoot(사다리 높이) 의 극한 평균이 $\beta$ 를 준다 (Asmussen–Glynn–Pitman 1995).
- $a$ 가 $\sqrt{\Delta t}$ 보다 충분히 크다: 근사 오차가 $o(\sqrt{\Delta t})$ 인 조건. $n=10$ 은 이 범위 밖이다.

### 2.6 직관

반사 원리는 "장벽에 닿은 순간부터 경로를 뒤집어도 확률이 같다" 는 대칭성 하나로 최댓값·도달시간의 분포를 공짜로 준다. 도달시간의 꼬리 $t^{-1/2}$ 는 너무 두꺼워 평균이 무한이다: **반드시 도달하지만 '언제' 의 평균은 없다** — 01x 의 1 차원 랜덤 워크 복귀시간과 같은 현상이다.

시뮬레이션 쪽 교훈: 컴퓨터는 격자점만 보므로 격자 사이의 봉우리를 놓친다. 그 편향은 $\sqrt{\Delta t}$ 로만 줄어서 $\Delta t$ 를 100 배 줄여야 편향이 10 배 준다. 대신 "격자점 사이는 브라운 다리" 라는 사실을 쓰면 봉우리 높이를 (분포적으로) 정확히 뽑을 수 있어 편향이 0 이 된다 — 같은 $\Delta t$, 비용은 두 배(난수 한 벌 더).

### 2.7 함정 (traps)

- $P(M_t\ge a)=2P(B_t\ge a)$ 는 $a>0$ 일 때. $a\le0$ 이면 확률 1 이다 (이 식은 1 을 넘는다).
- $\{T_a\le t\}$ 와 $\{B_t\ge a\}$ 는 다른 사건: 전자가 후자의 정확히 2 배 확률.
- $E[T_a]=\infty$ 인데 $T_a<\infty$ a.s.: "평균이 없다" 와 "도달하지 않는다" 를 혼동하지 말 것. 절단 평균 $E[T_a\wedge c]$ 는 $\sqrt{8c/\pi}$ 처럼 끝없이 자란다.
- 격자 최댓값은 항상 아래로, 격자 도달시간은 항상 위로 치우친다. `assert_close` 가 실패하면 난수 탓을 하기 전에 편향 $\beta\sqrt{\Delta t}$ 와 SE 를 비교하라 ($n=1000$, $N=20000$: 편향 $-0.009$ = $-2.7$ SE).
- 다리 보정은 "한 구간 안의 최댓값" 을 정확히 뽑지만 도달 *시각* 은 여전히 어느 구간인지만 안다 (오차 $\le\Delta t$). 시각이 필요하면 구간 왼쪽 끝을 쓰고 오차 한계를 SE 와 비교하라.
- 마지막 영점을 격자 부호변화로 추정하면 크게 편향된다 ($n=1000$ 에서 $\hat P(L_1\le1/4)\approx0.348$ vs $1/3$): 부호가 안 바뀐 구간에서도 확률 $e^{-2xy/\Delta t}$ 로 0 을 스친다. 양수 체류 비율 $A_1$ 은 격자 추정이 거의 편향 없다 ($+0.0007$).

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — 특히 격자 최댓값이 왜 *항상* 과소평가인지, 그리고 왜 격자를 조밀하게 하는 것보다 다리 보정이 나은지 한 문단으로.)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: $2(1-\Phi(\cdot))$, $\sqrt{2/\pi}$, $\Phi^{-1}(3/4)=0.6745$, $\Phi(1)=0.8413$, $\Phi(1.5)=0.9332$, 장벽 이동 $\beta\sqrt{\Delta t}$, $\frac2\pi\arcsin\sqrt x$.

In [ ]:
predictions = {
    # P(M_1 >= 1) 은?
    "p_max1_ge_1": None,
    # E[M_1] 은?
    "e_max1": None,
    # T_1 의 중앙값은? (평균은 얼마인가?)
    "median_T1": None,
    # P(M_1 >= 1, B_1 <= 0.5) 은?
    "p_joint": None,
    # 격자 n=100 (dt=0.01) 위 러닝 최댓값으로 추정한 P(M_1 >= 1) 은 참값 0.3173 에서 얼마나 벗어날까? (값을 적으세요)
    "p_grid_n100": None,
    # [0,1] 에서 양수 체류 시간 비율 A_1 에 대해 P(A_1 <= 1/4) 은?
    "p_arcsine_quarter": None,
}

## 4. Simulate — 시뮬레이션

먼저 이 노트북에서만 쓰는 도우미를 정의한다. `bridge_step_max` 는 2.1 의 역변환 공식으로 **각 격자 구간의 정확한 최댓값**을 뽑고, `bridge_zero_cross` 는 구간 안에서 0 을 지났는지의 지시함수를 뽑는다. `max_hit_stats` 는 경로를 `chunk` 개씩 만들어 경로별 스칼라만 보관한다 (메모리 $\approx$ 몇 $\times$ chunk $\times$ n_steps $\times$ 8 B). `drift` 인자는 E3 용이다 — 구간 다리 최댓값 공식은 양 끝점만 쓰므로 드리프트와 무관하다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def bridge_step_max(B, dt, rng):
    """격자 구간 [t_i, t_{i+1}] 각각의 정확한 최댓값 표본 (양 끝점 조건부 브라운 다리; shape (n_paths, n_steps))."""
    x, y = B[:, :-1], B[:, 1:]
    U = rng.random(x.shape)
    return 0.5 * (x + y + np.sqrt((y - x) ** 2 - 2 * dt * np.log(U)))

def bridge_zero_cross(B, dt, rng):
    """각 격자 구간에서 연속 경로가 0 을 지나는지의 지시함수 (같은 부호일 때 확률 exp(-2xy/dt))."""
    x, y = B[:, :-1], B[:, 1:]
    return (x * y <= 0) | (rng.random(x.shape) < np.exp(-2 * x * y / dt))

def max_hit_stats(n_paths, n_steps, T, rng, level=1.0, chunk=2000, drift=0.0):
    """경로별 스칼라 통계 (격자 최댓값, 다리 보정 최댓값, 끝값, 도달 구간, 양수 비율, 마지막 영점) 의 dict."""
    dt, idx = T / n_steps, np.arange(n_steps)
    keys = ("M_grid", "M_exact", "B_T", "M_exact_half", "hit_idx", "frac_pos", "L_grid", "L_exact")
    out, done = {k: [] for k in keys}, 0
    while done < n_paths:
        m = min(chunk, n_paths - done)
        t, B = brownian_paths(m, n_steps, T, rng)
        B = B + drift * t
        Ms = bridge_step_max(B, dt, rng)                  # 구간별 최댓값
        Mex = np.maximum.accumulate(Ms, axis=1)          # 격자 시각 (k+1)dt 까지의 정확한 러닝 최댓값 (열 k)
        hit = Ms >= level
        out["M_grid"].append(running_max(B)[:, -1]);   out["M_exact"].append(Mex[:, -1])
        out["B_T"].append(B[:, -1]);                    out["M_exact_half"].append(Mex[:, n_steps // 2 - 1])
        out["hit_idx"].append(np.where(hit.any(1), hit.argmax(1), -1))
        out["frac_pos"].append((B[:, 1:] > 0).mean(1))
        out["L_grid"].append(((B[:, :-1] * B[:, 1:] <= 0) * idx).max(1) * dt)
        out["L_exact"].append((bridge_zero_cross(B, dt, rng) * idx).max(1) * dt)
        done += m
    return {k: np.concatenate(v) for k, v in out.items()}

닫힌 형식용 작은 함수들: Lévy 밀도, 아크사인 밀도, 절단 평균 $E[T_a\wedge c]=\int_0^c P(T_a>s)\,ds$ (수치적분), 그리고 BGK 연속성 보정 예측.

In [ ]:
def levy_pdf(t, a=1.0):
    """T_a 의 밀도 a / sqrt(2 pi t^3) exp(-a^2 / 2t)."""
    return a / np.sqrt(2 * np.pi * t**3) * np.exp(-a**2 / (2 * t))

def arcsine_pdf(x):
    return 1 / (np.pi * np.sqrt(x * (1 - x)))

def e_min_T(c, a=1.0):
    """E[min(T_a, c)] = int_0^c (2 Phi(a/sqrt(s)) - 1) ds."""
    return quad(lambda s: 2 * norm.cdf(a / np.sqrt(s)) - 1, 0, c, limit=200)[0]

def bgk_corrected_p(a, dt, T=1.0, beta=BETA):
    """격자 최댓값의 P(M^(n)_T >= a) 예측: 장벽을 beta sqrt(dt) 만큼 민다."""
    return 2 * (1 - norm.cdf((a + beta * np.sqrt(dt)) / np.sqrt(T)))

p_true = 2 * (1 - norm.cdf(1.0))
print(f"P(M_1 >= 1) = 2(1 - Phi(1)) = {p_true:.6f};  BGK-corrected grid predictions:",
      {n: round(bgk_corrected_p(1.0, 1 / n), 4) for n in N_STEPS_LIST})

### 4.1 반사 원리 그림 (Fig 1)

경로 200 개 ($n=1000$, $T=1$) 중 격자 위에서 $a=1$ 에 닿은 것 3 개를 골라, 원 경로와 도달 시점 이후를 $2a-B_t$ 로 뒤집은 반사 경로를 겹쳐 그린다. `first_passage_index` 가 주는 도달 인덱스는 06b 식 **격자 근사** $T^{(n)}_a\ge T_a$ 라는 점을 기억하자 — 실제 경로는 그 직전 구간 어딘가에서 이미 1 에 닿았을 수 있다.

In [ ]:
A = 1.0
t1, B1 = brownian_paths(200, 1000, 1.0, rng)
idx1 = first_passage_index(B1, A)
hitters = np.flatnonzero(idx1 >= 0)
n_hit = mc_proportion(idx1 >= 0)
print(f"paths hitting a=1 on the grid: {n_hit}   (2(1-Phi(1)) = {p_true:.4f}; grid estimate is biased low)")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, k in zip(axes, hitters[:3]):
    Br = B1[k].copy(); Br[idx1[k]:] = 2 * A - B1[k, idx1[k]:]
    ax.plot(t1, B1[k], color="C0", lw=1.0, label="path $B_t$")
    ax.plot(t1[idx1[k]:], Br[idx1[k]:], color="C3", lw=1.0, ls="--", label="reflected $2a - B_t$")
    ax.axhline(A, color="k", lw=0.8); ax.plot(t1[idx1[k]], A, "ko", ms=5)
    ax.set_title(f"path {k}: grid hit at t = {t1[idx1[k]]:.3f}", fontsize=10); ax.set_xlabel("t")
axes[0].set_ylabel("B_t"); axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Fig 1. Reflection principle: after $T_a$ the mirrored path is another Brownian motion  (a = 1)")
plt.tight_layout(); plt.show()

점(도달 시각) 이후의 파란 경로와 빨간 경로는 서로 거울상이고, 강마르코프 + 대칭성에 의해 **같은 확률**을 갖는다. 끝값이 $a$ 아래인 경로마다 끝값이 $a$ 위인 짝이 있으므로 $P(T_a\le t)=2P(B_t>a)$ — 정리 1 의 그림이다.

### 4.2 격자 편향 측정 vs 다리 보정 (Fig 2)

$n\in$ `N_STEPS_LIST` 각각에 대해 $N$ 개 경로의 격자 최댓값 $M^{(n)}_1$ 과 다리 보정 최댓값 $M_1$ 을 뽑아 $\hat P(M_1\ge1)$ 과 $\hat E[M_1]$ 을 잰다. 참값은 $2(1-\Phi(1))=0.3173$, $\sqrt{2/\pi}=0.7979$; 격자 쪽 예측은 정리 4 의 장벽 이동 $\beta\sqrt{\Delta t}$. $n=1000$ 의 결과는 4.3 과 4.6 에서 다시 쓴다.

In [ ]:
e_true = np.sqrt(2 / np.pi)
stats = {}
print("     n      dt   P_grid   BGK-pred  P_bridge  (true {:.4f})  |  E[M]_grid  pred    E[M]_bridge (true {:.4f})".format(p_true, e_true))
for n in N_STEPS_LIST:
    S = max_hit_stats(N_PATHS, n, 1.0, rng, level=1.0, chunk=2000 if n <= 4000 else 400)
    dt = 1.0 / n
    S["p_grid"], S["p_ex"] = mc_proportion(S["M_grid"] >= 1), mc_proportion(S["M_exact"] >= 1)
    S["m_grid"], S["m_ex"] = mc_estimate(S["M_grid"]), mc_estimate(S["M_exact"])
    stats[n] = S
    print(f"{n:6d}  {dt:6.4f}   {S['p_grid'].mean:.4f}   {bgk_corrected_p(1.0, dt):.4f}    {S['p_ex'].mean:.4f}                  |   "
          f"{S['m_grid'].mean:.4f}    {e_true - BETA * np.sqrt(dt):.4f}   {S['m_ex'].mean:.4f}")
S1000 = stats[1000]
print(f"SE of each proportion ~ {S1000['p_grid'].se:.4f}, of each mean ~ {S1000['m_grid'].se:.4f}  (N = {N_PATHS})")

In [ ]:
dts = np.array([1.0 / n for n in N_STEPS_LIST])
pg = np.array([stats[n]["p_grid"].mean for n in N_STEPS_LIST]); pg_se = np.array([stats[n]["p_grid"].se for n in N_STEPS_LIST])
pe = np.array([stats[n]["p_ex"].mean for n in N_STEPS_LIST]);   pe_se = np.array([stats[n]["p_ex"].se for n in N_STEPS_LIST])
bias = e_true - np.array([stats[n]["m_grid"].mean for n in N_STEPS_LIST]); bias_se = np.array([stats[n]["m_grid"].se for n in N_STEPS_LIST])
dt_fine = np.logspace(np.log10(dts.min()) - 0.3, np.log10(dts.max()) + 0.3, 100)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.2))
axL.errorbar(dts, pg, yerr=2 * pg_se, fmt="o-", color="C0", capsize=3, label="grid max $M^{(n)}_1$")
axL.errorbar(dts, pe, yerr=2 * pe_se, fmt="s-", color="C2", capsize=3, label="bridge-corrected $M_1$")
axL.plot(dt_fine, bgk_corrected_p(1.0, dt_fine), "k:", lw=1.2, label="BGK: $2(1-\\Phi(1+\\beta\\sqrt{\\Delta t}))$")
axL.axhline(p_true, color="k", ls="--", lw=1, label=f"exact 2(1-Phi(1)) = {p_true:.4f}")
axL.set_xscale("log"); axL.invert_xaxis(); axL.set_xlabel("$\\Delta t$ (log, finer to the right)"); axL.set_ylabel("$\\hat P(M_1 \\geq 1)$")
axL.legend(fontsize=8); axL.set_title("Fig 2a. Grid estimate is biased low, bridge is not")
axR.errorbar(dts, bias, yerr=2 * bias_se, fmt="o", color="C0", capsize=3, label="$\\sqrt{2/\\pi} - \\hat E[M^{(n)}_1]$")
axR.plot(dt_fine, BETA * np.sqrt(dt_fine), "k--", lw=1, label="$\\beta\\sqrt{\\Delta t}$, slope 1/2")
axR.set_xscale("log"); axR.set_yscale("log"); axR.set_xlabel("$\\Delta t$ (log)"); axR.set_ylabel("bias of E[M] (log)")
axR.legend(fontsize=8); axR.set_title("Fig 2b. Bias shrinks only like $\\sqrt{\\Delta t}$")
plt.tight_layout(); plt.show()

왼쪽: 격자 추정은 $\Delta t$ 가 100 배 작아질 때마다 참값에 10 배 가까워지고 (BGK 점선을 따라간다), 다리 보정은 $\Delta t$ 와 무관하게 참값 주위에 수평이다. 오른쪽: log-log 에서 편향이 기울기 $1/2$ 의 직선 $\beta\sqrt{\Delta t}$ 위에 놓인다 — $n=10$ 은 근사 범위 밖이라 약간 벗어난다. 에러바는 $\pm2$ SE.

### 4.3 다리 보정 추정량으로 반사 원리 검증 (Fig 3)

$n=1000$ ($\Delta t=10^{-3}$) 의 배열로 정리 1 의 네 숫자를 잰다: $P(M_1\ge1)$, $E[M_1]$, 결합확률 $P(M_1\ge1,B_1\le0.5)=1-\Phi(1.5)$, 그리고 $P(T_1\le0.5)=P(M_{0.5}\ge1)=2(1-\Phi(\sqrt2))$ — 마지막 것은 처음 500 개 구간의 다리 최댓값의 max (`M_exact_half`, 열 인덱스 $n/2-1$) 로 정확히 $t=0.5$ 까지의 최댓값이다.

In [ ]:
p_max_exact_hat = S1000["p_ex"]
e_max_exact_hat = S1000["m_ex"]
p_joint_hat = mc_proportion((S1000["M_exact"] >= 1) & (S1000["B_T"] <= 0.5))
p_T1_half_hat = mc_proportion(S1000["M_exact_half"] >= 1)
p_max_grid_n100_hat, p_max_grid_n1000_hat = stats[100]["p_grid"], stats[1000]["p_grid"]
print(f"P(M_1 >= 1)            bridge = {p_max_exact_hat}   exact {p_true:.6f}")
print(f"E[M_1]                 bridge = {e_max_exact_hat}   exact {e_true:.6f}")
print(f"P(M_1 >= 1, B_1 <= .5) bridge = {p_joint_hat}   exact {1 - norm.cdf(1.5):.6f}")
print(f"P(T_1 <= 0.5)          bridge = {p_T1_half_hat}   exact {2 * (1 - norm.cdf(np.sqrt(2))):.6f}")
for n in (100, 1000):
    z = (stats[n]["p_grid"].mean - p_true) / stats[n]["p_grid"].se
    print(f"grid n={n:5d}: P(M_1 >= 1) = {stats[n]['p_grid']}  ->  {z:+.1f} SE from the truth (expected bias {bgk_corrected_p(1.0, 1 / n) - p_true:+.4f})")

In [ ]:
xg = np.linspace(0, 3.5, 300)
fig, ax = plt.subplots(figsize=(8, 4.2))
plot_hist_vs_pdf(S1000["M_exact"], xg, 2 * norm.pdf(xg), ax=ax, bins=np.linspace(0, 3.5, 50), label_pdf="half-normal $2\\phi(x)$")
ax.hist(S1000["M_grid"], bins=np.linspace(0, 3.5, 50), density=True, histtype="step", color="C3", lw=1.3, label="grid max $M^{(1000)}_1$ (shifted left)")
ax.axvline(e_true, color="k", ls="--", lw=1, label=f"E[M_1] = sqrt(2/pi) = {e_true:.4f}")
ax.set_xlabel("$M_1$"); ax.set_ylabel("density"); ax.legend(fontsize=8)
ax.set_title(f"Fig 3. Running maximum on [0, 1]: bridge-corrected vs grid  (N = {N_PATHS}, n = 1000)")
plt.show()

다리 보정 히스토그램은 반정규 밀도 $2\phi(x)$ 에 올라탄다 ($M_1\overset d=|B_1|$). 격자 히스토그램(빨간 계단)은 0 근처가 부풀고 오른쪽이 얇다 — 평균이 $\beta\sqrt{\Delta t}\approx0.018$ 만큼 왼쪽으로 밀린 것이다.

### 4.4 도달시간의 분포 (Fig 4)

구간 $[0,4]$, $n=4000$ ($\Delta t=10^{-3}$). 다리 최댓값이 처음 1 을 넘는 구간의 **왼쪽 끝**을 $\hat T_1$ 로 쓴다 (편향 $\in[-\Delta t,0]$, 함정 5). $T_1>4$ 인 경로는 절단된다 (질량 $2\Phi(0.5)-1=0.383$). 비교 대상: Lévy 밀도, $P(T_1\le4)=2(1-\Phi(0.5))$, 절단 평균 $E[T_1\wedge4]$ (수치적분), 중앙값 $2.198$ (중앙값 $<4$ 이므로 절단이 영향 없음). 중앙값의 SE 는 $1/(2f(m)\sqrt N)$ 로 직접 넣는다.

In [ ]:
T4, N4 = 4.0, 4000
S4 = max_hit_stats(N_PATHS, N4, T4, rng, level=1.0, chunk=1000)
dt4 = T4 / N4
T1 = np.where(S4["hit_idx"] >= 0, S4["hit_idx"] * dt4, np.inf)
T1c = np.minimum(T1, T4)                                  # 4 에서 절단
p_T1_le4_hat = mc_proportion(T1 <= T4)
e_minT4_hat = mc_estimate(T1c)
median_hat = float(np.median(T1c))
median_exact = 1 / norm.ppf(0.75) ** 2
median_se = 1 / (2 * levy_pdf(median_exact) * np.sqrt(N_PATHS))
T1_exact_sample = np.minimum(1 / rng.standard_normal(N_PATHS) ** 2, T4)   # 정확 표본기 a^2/Z^2, 같은 절단
print(f"P(T_1 <= 4)    = {p_T1_le4_hat}   exact 2(1-Phi(0.5)) = {2 * (1 - norm.cdf(0.5)):.6f}")
print(f"E[min(T_1, 4)] = {e_minT4_hat}   exact (quad) = {e_min_T(4.0):.5f}   (left-endpoint bias in [-{dt4}, 0])")
print(f"median(T_1)    = {median_hat:.4f} ± {median_se:.4f}   exact 1/Phi^-1(3/4)^2 = {median_exact:.6f}")

In [ ]:
bins4 = np.arange(0, T4 + 0.05, 0.1)
tg = np.linspace(0.02, T4, 400)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.2))
axL.hist(T1[T1 <= T4], bins=bins4, weights=np.full((T1 <= T4).sum(), 1 / (N_PATHS * 0.1)), alpha=0.5, label="simulated $T_1$ (bridge, censored at 4)")
axL.plot(tg, levy_pdf(tg), color="C1", lw=2, label="Levy density $\\frac{1}{\\sqrt{2\\pi t^3}}e^{-1/2t}$")
axL.axvline(median_exact, color="k", ls="--", lw=1, label=f"median = {median_exact:.3f}")
axL.set_xlabel("t"); axL.set_ylabel("density (total mass 1, incl. $P(T_1>4)$)"); axL.legend(fontsize=8)
axL.set_title(f"Fig 4a. Hitting time of level 1  (N = {N_PATHS})")
axR.plot(np.sort(T1_exact_sample), np.sort(T1c), ".", ms=2, alpha=0.5)
axR.plot([0, T4], [0, T4], "k--", lw=1, label="y = x")
axR.set_xlabel("exact sampler $1/Z^2$ (censored at 4)"); axR.set_ylabel("simulated $\\hat T_1$ (censored at 4)"); axR.legend()
axR.set_title("Fig 4b. QQ plot against the exact sampler")
plt.tight_layout(); plt.show()

왼쪽: 밀도는 0 근처에서 급히 솟았다가 $t^{-3/2}$ 꼬리로 천천히 내려온다 — 4 까지 봐도 질량의 38% 가 아직 밖에 있다. 오른쪽: 정확 표본기 $1/Z^2$ 와 QQ 점들이 대각선 위 — $T_1\overset d=1/Z^2$ 이고 다리 보정 격자 추정 $\hat T_1$ 의 편향은 $\Delta t$ 이내다.

### 4.5 $E[T_a]=\infty$ 를 눈으로 (Fig 5)

정확 표본기 $T=1/Z^2$ 로 $N$ 개를 뽑아 (a) 러닝 표본평균 $\bar T_k=\frac1k\sum_{i\le k}T_i$ 를 그리고, (b) 절단 평균 $E[T\wedge c]$ 를 $c=1,\dots,10^4$ 에서 수치적분과 비교하며, (c) 꼬리 $P(T>t)$ 를 $\sqrt{2/\pi}\,t^{-1/2}$ 와 log-log 로 비교한다.

In [ ]:
T_ex = 1 / rng.standard_normal(N_EXACT) ** 2
running_mean = np.cumsum(T_ex) / np.arange(1, N_EXACT + 1)
c_list = [1, 10, 100, 1000, 10_000]
trunc = {c: mc_estimate(np.minimum(T_ex, c)) for c in c_list}
e_minT100_hat = trunc[100]
print(f"running mean at N = {N_EXACT}: {running_mean[-1]:.2f}   (largest single sample {T_ex.max():.3g}, sample sum ~ N^2)")
print("     c    E[min(T,c)] MC             quad      sqrt(8c/pi)")
for c in c_list:
    print(f"{c:6d}    {trunc[c]!s:28s}  {e_min_T(c):9.4f}    {np.sqrt(8 * c / np.pi):8.3f}")

In [ ]:
tt = np.logspace(-1, 4, 200)
tail_hat = 1 - np.searchsorted(np.sort(T_ex), tt, side="right") / N_EXACT
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.2))
kk = np.unique(np.logspace(0, np.log10(N_EXACT), 2000).astype(int)) - 1
axL.plot(kk + 1, running_mean[kk], lw=1.2)
axL.set_xscale("log"); axL.set_xlabel("sample size k (log)"); axL.set_ylabel("running mean of $T_1 = 1/Z^2$")
axL.set_title("Fig 5a. The sample mean never settles (E[T_1] = inf)")
axR.loglog(tt, tail_hat, lw=1.5, label="empirical $P(T_1 > t)$")
axR.loglog(tt, 2 * norm.cdf(1 / np.sqrt(tt)) - 1, "C1--", lw=1.2, label="exact $2\\Phi(t^{-1/2}) - 1$")
axR.loglog(tt, np.sqrt(2 / np.pi) / np.sqrt(tt), "k:", lw=1.2, label="$\\sqrt{2/\\pi}\\, t^{-1/2}$")
axR.set_xlabel("t (log)"); axR.set_ylabel("tail probability (log)"); axR.legend(fontsize=8)
axR.set_title("Fig 5b. Tail decays like $t^{-1/2}$: too fat for a mean")
plt.tight_layout(); plt.show()

(a) 러닝 평균은 계단을 오르며 끝없이 자란다: 가끔 나오는 거대한 표본 하나가 그때까지의 합을 압도한다 ($Z\approx0$ 이면 $1/Z^2$ 은 얼마든지 크다). (b) 절단 평균은 $\sqrt{8c/\pi}$ 처럼 $c$ 의 제곱근으로 자라 한계가 없다. (c) 꼬리 기울기 $-1/2$: $\int t^{-1/2}dt$ 가 발산하는 바로 그 꼬리.

### 4.6 아크사인 법칙 예고 (Fig 6)

$n=1000$ 의 배열을 다시 쓴다. `frac_pos` 는 격자점 중 $B>0$ 인 비율 (격자 편향 $+0.0007$, 무시). 마지막 영점은 두 가지로 잰다: `L_grid` (마지막 격자 부호변화 구간의 왼쪽 끝) 와 `L_exact` (다리 교차 확률 $e^{-2xy/\Delta t}$ 로 부호가 안 바뀐 구간의 교차까지 잡은 마지막 교차 구간의 왼쪽 끝).

In [ ]:
p_arcsine_A_hat = mc_proportion(S1000["frac_pos"] <= 0.25)
p_arcsine_Lgrid_hat = mc_proportion(S1000["L_grid"] <= 0.25)
p_arcsine_Lexact_hat = mc_proportion(S1000["L_exact"] <= 0.25)
p_arc = 2 / np.pi * np.arcsin(np.sqrt(0.25))
print(f"P(A_1 <= 1/4)  grid                = {p_arcsine_A_hat}   exact (2/pi) asin(1/2) = {p_arc:.6f}")
print(f"P(L_1 <= 1/4)  grid sign changes   = {p_arcsine_Lgrid_hat}   ->  {(p_arcsine_Lgrid_hat.mean - p_arc) / p_arcsine_Lgrid_hat.se:+.1f} SE (biased: missed crossings)")
print(f"P(L_1 <= 1/4)  bridge crossings    = {p_arcsine_Lexact_hat}")

In [ ]:
xa = np.linspace(0.002, 0.998, 400)
bins_a = np.linspace(0, 1, 41)
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.2))
plot_hist_vs_pdf(S1000["frac_pos"], xa, arcsine_pdf(xa), ax=axL, bins=bins_a, label_pdf="arcsine $1/(\\pi\\sqrt{x(1-x)})$")
axL.set_xlabel("$A_1$ = fraction of time positive"); axL.set_ylabel("density"); axL.set_ylim(0, 4)
axL.set_title("Fig 6a. Time spent positive (n = 1000 grid)")
axR.hist(S1000["L_grid"], bins=bins_a, density=True, histtype="step", color="C3", lw=1.3, label="last grid sign change (biased)")
axR.hist(S1000["L_exact"], bins=bins_a, density=True, alpha=0.5, label="last bridge crossing")
axR.plot(xa, arcsine_pdf(xa), color="C1", lw=2, label="arcsine density")
axR.set_xlabel("$L_1$ = last zero in [0, 1]"); axR.set_ylabel("density"); axR.set_ylim(0, 4); axR.legend(fontsize=8)
axR.set_title("Fig 6b. Last zero: grid vs bridge-corrected")
plt.tight_layout(); plt.show()

U 자형: 경로는 대부분의 시간을 한쪽 부호에서 보내고 (비율이 0 이나 1 근처), 마지막 영점도 아주 이르거나 아주 늦다. 격자 부호변화(빨간 계단)는 왼쪽으로 치우친다 — 부호가 안 바뀐 구간에서 0 을 스친 경우를 놓쳐 마지막 영점을 **너무 이르게** 잡기 때문이다. 다리 교차 보정은 아크사인 밀도에 맞는다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드에서 계산한다: $2(1-\Phi(\cdot))$, $\sqrt{2/\pi}$, $1-\Phi(1.5)$, $1/\Phi^{-1}(3/4)^2$, 수치적분 $E[T_1\wedge c]$, $\frac2\pi\arcsin\frac12$, BGK 보정 $2(1-\Phi(1+\beta\sqrt{\Delta t}))$.

In [ ]:
exact = {
    "p_max": p_true,                                              # 2(1 - Phi(1))
    "e_max": e_true,                                              # sqrt(2/pi)
    "p_joint": 1 - norm.cdf(1.5),                                 # P(B_1 >= 2*1 - 0.5)
    "p_T1_half": 2 * (1 - norm.cdf(1 / np.sqrt(0.5))),            # 2(1 - Phi(sqrt 2))
    "p_T1_le4": 2 * (1 - norm.cdf(1 / np.sqrt(4.0))),             # 2(1 - Phi(0.5))
    "e_minT4": e_min_T(4.0),
    "median": median_exact,
    "e_minT100": e_min_T(100.0),
    "p_arc": p_arc,                                               # 1/3
    "bgk_100": bgk_corrected_p(1.0, 1 / 100),
    "bgk_1000": bgk_corrected_p(1.0, 1 / 1000),
}
rows = [
    {"name": "P(M_1 >= 1), bridge-corrected, n=1000", "predicted": predictions["p_max1_ge_1"], "estimate": p_max_exact_hat, "exact": exact["p_max"]},
    {"name": "P(M_1 >= 1), grid max, n=100  [bias demo]", "predicted": predictions["p_grid_n100"], "estimate": p_max_grid_n100_hat, "exact": exact["p_max"]},
    {"name": "  same estimate vs BGK-corrected value", "predicted": None, "estimate": p_max_grid_n100_hat, "exact": exact["bgk_100"]},
    {"name": "P(M_1 >= 1), grid max, n=1000  [borderline]", "predicted": None, "estimate": p_max_grid_n1000_hat, "exact": exact["p_max"]},
    {"name": "  same estimate vs BGK-corrected value", "predicted": None, "estimate": p_max_grid_n1000_hat, "exact": exact["bgk_1000"]},
    {"name": "E[M_1], bridge-corrected", "predicted": predictions["e_max1"], "estimate": e_max_exact_hat, "exact": exact["e_max"]},
    {"name": "P(M_1 >= 1, B_1 <= 0.5)", "predicted": predictions["p_joint"], "estimate": p_joint_hat, "exact": exact["p_joint"]},
    {"name": "P(T_1 <= 0.5)", "predicted": None, "estimate": p_T1_half_hat, "exact": exact["p_T1_half"]},
    {"name": "P(T_1 <= 4)", "predicted": None, "estimate": p_T1_le4_hat, "exact": exact["p_T1_le4"]},
    {"name": "E[min(T_1, 4)]", "predicted": None, "estimate": e_minT4_hat, "exact": exact["e_minT4"]},
    {"name": "median(T_1)", "predicted": predictions["median_T1"], "estimate": median_hat, "se": median_se, "exact": exact["median"]},
    {"name": "E[min(T_1, 100)] (exact sampler)", "predicted": None, "estimate": e_minT100_hat, "exact": exact["e_minT100"]},
    {"name": "P(A_1 <= 1/4) (time positive)", "predicted": predictions["p_arcsine_quarter"], "estimate": p_arcsine_A_hat, "exact": exact["p_arc"]},
    {"name": "P(L_1 <= 1/4), grid sign changes  [bias demo]", "predicted": None, "estimate": p_arcsine_Lgrid_hat, "exact": exact["p_arc"]},
    {"name": "P(L_1 <= 1/4), bridge crossings", "predicted": None, "estimate": p_arcsine_Lexact_hat, "exact": exact["p_arc"]},
]
Markdown(compare_table(rows))

각 행이 어긋난다면 무엇이 의심되는가:
- **P(M_1 >= 1) bridge**: 다리 최댓값 공식의 부호/$\Delta t$ 가 틀렸거나 (예: $\ln U$ 앞의 $-2\Delta t$), `np.maximum.accumulate` 를 안 했다.
- **grid n=100 / n=1000 vs 참값**: 'investigate' 가 **정답**이다 — 이것이 격자 편향 (FAST 모드에서는 SE 가 커서 편향이 잡음 아래 숨을 수 있다; 그래서 위험하다). 이 두 행은 assert 하지 않는다.
- **grid vs BGK 보정값**: 어긋나면 $\beta$ 나 $\sqrt{\Delta t}$ 를 잘못 넣었다 (보정 잔차는 $O(\Delta t)$ 로 SE 보다 훨씬 작다).
- **E[M_1]**: 히스토그램이 반정규가 아니다 — 최댓값 대신 끝값 $|B_1|$ 의 절반만 봤거나 음수 최댓값(항상 $\ge0$ 이어야 함) 이 섞였다.
- **P(M_1 >= 1, B_1 <= 0.5)**: $2a-b$ 를 $a-b$ 로 착각 (반사는 수준 $a$ 기준).
- **P(T_1 <= 0.5)**: `M_exact_half` 의 열 인덱스가 $n/2-1$ 이 아니다 (열 $k$ 는 구간 $k+1$ 개 = 시각 $(k+1)\Delta t$ 까지).
- **P(T_1 <= 4), E[min(T_1,4)]**: 도달 구간을 다리 최댓값이 아니라 격자값으로 잡았다 (과대 편향) 또는 절단을 잊었다.
- **median(T_1)**: SE 는 `mc_estimate` 가 아니라 $1/(2f(m)\sqrt N)$ 이다; 절단 표본이라도 중앙값 $<4$ 이면 영향 없다.
- **E[min(T_1,100)]**: 정확 표본기가 $1/Z^2$ 이 아니라 $1/Z$ 나 $1/|Z|$ 다.
- **P(A_1 <= 1/4)**: 격자 오른쪽 끝점 대신 $B_0=0$ 을 포함시켰다 (작은 편향) — 큰 차이면 아크사인 표본이 아니다.
- **P(L_1 <= 1/4) grid**: 'investigate' 가 예상 결과 (+4.5 SE, $N=20000$); assert 하지 않는다.
- **P(L_1 <= 1/4) bridge**: 교차 확률 $e^{-2xy/\Delta t}$ 의 부호가 틀렸다 ($xy>0$ 일 때 지수는 음수여야).

In [ ]:
assert_close(p_max_exact_hat, exact["p_max"], k=4, name="P(M_1>=1) bridge-corrected n=1000")
assert_close(p_max_grid_n100_hat, exact["bgk_100"], k=4, name="P(M_1>=1) grid n=100 vs continuity-corrected")
assert_close(p_max_grid_n1000_hat, exact["bgk_1000"], k=4, name="P(M_1>=1) grid n=1000 vs continuity-corrected")
assert_close(e_max_exact_hat, exact["e_max"], k=4, name="E[M_1] bridge-corrected")
assert_close(p_joint_hat, exact["p_joint"], k=4, name="P(M_1>=1, B_1<=0.5)")
assert_close(p_T1_half_hat, exact["p_T1_half"], k=4, name="P(T_1<=0.5)")
assert_close(p_T1_le4_hat, exact["p_T1_le4"], k=4, name="P(T_1<=4)")
assert_close(e_minT4_hat, exact["e_minT4"], k=4, name="E[min(T_1,4)]")
assert_close(median_hat, exact["median"], se=median_se, k=4, name="median T_1")
assert_close(e_minT100_hat, exact["e_minT100"], k=4, name="E[min(T_1,100)] exact sampler")
assert_close(p_arcsine_A_hat, exact["p_arc"], k=4, name="P(A_1<=1/4)")
assert_close(p_arcsine_Lexact_hat, exact["p_arc"], k=4, name="P(L_1<=1/4) bridge-corrected")
# 일부러 assert 하지 않는 행: grid n=100/1000 vs 참값, L_grid — 편향 시연이다. z 값만 출력한다.
for name, est, ex, bias in [("grid n=100 vs truth", p_max_grid_n100_hat, exact["p_max"], exact["bgk_100"] - p_true),
                            ("grid n=1000 vs truth", p_max_grid_n1000_hat, exact["p_max"], exact["bgk_1000"] - p_true),
                            ("L_grid vs 1/3", p_arcsine_Lgrid_hat, exact["p_arc"], 0.348 - 1 / 3)]:   # 0.348: n=1000 격자 부호변화의 N=200000 사전 측정값 (닫힌 형식 아님)
    print(f"{name:22s}: z = {(est.mean - ex) / est.se:+.1f} SE   (expected bias {bias:+.4f} = {bias / est.se:+.1f} SE at this N; the rest is noise)")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

손으로: (a) $P(M_2\ge3)$; (b) $P(M_1\ge1,\ B_1\le0)$; (c) $M_t$ 의 밀도와 $E[M_t]$; (d) $T_a$ 의 중앙값이 $a^2$ 에 비례하는 이유를 자기유사성으로 설명하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $P(M_2\ge3)=2\big(1-\Phi(3/\sqrt2)\big)=2(1-\Phi(2.1213))=0.033895$.

(b) $b=0\le a=1$ 이므로 $P(M_1\ge1,B_1\le0)=P(B_1\ge2)=1-\Phi(2)=0.022750$. (검산: $P(M_1\ge1)=0.3173$ 의 절반 $0.1587$ 은 $P(M_1\ge1,B_1\le1)$ 이고, 그중 $B_1\le0$ 인 부분은 훨씬 작다.)

(c) $M_t\overset d=|B_t|=\sqrt t|Z|$: 밀도 $\frac2{\sqrt t}\phi(x/\sqrt t)$ ($x\ge0$), $E[M_t]=\sqrt t\,E|Z|=\sqrt{2t/\pi}=0.7979\sqrt t$.

(d) $B'_s=B_{a^2s}/a$ 는 표준 BM 이다. $T_a=\inf\{t:B_t=a\}=\inf\{a^2s:B_{a^2s}=a\}=a^2\inf\{s:B'_s=1\}\overset d=a^2T_1$. 분포 전체가 $a^2$ 배로 늘어나므로 중앙값도 $a^2\cdot2.198$.

```python
print(2 * (1 - norm.cdf(3 / np.sqrt(2))), 1 - norm.cdf(2), np.sqrt(2 / np.pi))   # 0.033895 0.022750 0.797885
```

</details>

### E2 유도 후 시뮬레이션 검증

(i) $T_a$ 의 밀도를 CDF 미분으로 유도하고 $E[T_a]=\infty$ 를 꼬리 적분으로 보여라. (ii) $T_a\overset d=a^2/Z^2$ 를 보여라. (iii) $P(1\le T_1\le4)$ 를 계산하고, 정확 표본기 `1/Z**2` (`N_EXACT` 개) 와 4.4 의 다리 보정 격자 `T1` 두 방법으로 `mc_proportion` 을 구해 `assert_close` 하라. $P(T_1>4)$ 도 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(i), (ii) 본문 정리 2 의 증명. 핵심 한 줄: $P(T_a>t)=2\Phi(a/\sqrt t)-1\approx a\sqrt{2/\pi}\,t^{-1/2}$ 이고 $\int^\infty t^{-1/2}dt=\infty$.

(iii) $P(1\le T_1\le4)=P(1\le1/Z^2\le4)=P(\tfrac12\le|Z|\le1)=2\big(\Phi(1)-\Phi(0.5)\big)=0.299765$. $P(T_1>4)=P(|Z|<\tfrac12)=2\Phi(0.5)-1=0.382925$. 정확 표본기의 SE 는 $\approx0.0005$ ($N=10^6$), 격자 쪽은 $\approx0.0032$ ($N=20000$); 격자 $\hat T_1$ 은 왼쪽 끝을 쓰므로 편향이 $f(1)\Delta t+f(4)\Delta t\approx0.0003$ 으로 SE 보다 훨씬 작다.

```python
p_mid = 2 * (norm.cdf(1) - norm.cdf(0.5))
p_gt4 = 2 * norm.cdf(0.5) - 1
Z = rng.standard_normal(N_EXACT)
p_mid_sampler = mc_proportion((1 / Z**2 >= 1) & (1 / Z**2 <= 4))
p_mid_grid = mc_proportion((T1 >= 1) & (T1 <= 4))
p_gt4_grid = mc_proportion(T1 > 4)
print(p_mid_sampler, p_mid_grid, p_mid)      # ~0.2998 ± 0.0005, ~0.300 ± 0.003, 0.299765
print(p_gt4_grid, p_gt4)                     # ~0.383 ± 0.003, 0.382925
assert_close(p_mid_sampler, p_mid, k=4, name="P(1<=T_1<=4) sampler")
assert_close(p_mid_grid, p_mid, k=4, name="P(1<=T_1<=4) bridge grid")
assert_close(p_gt4_grid, p_gt4, k=4, name="P(T_1>4)")
```

</details>

### E3 가정을 깨보기 — 대칭성

드리프트 BM $X_t=0.5t+B_t$ 에 대해 반사 원리를 그대로 적용하면 $P(\max_{s\le1}X_s\ge1)=2P(X_1\ge1)=2(1-\Phi(0.5))=0.617$ 이라고 주장하게 된다. `max_hit_stats(..., drift=0.5)` 로 $\hat P$ 를 구하고 (구간 다리 최댓값 공식은 양 끝점만 쓰므로 드리프트와 무관하게 그대로 맞다), 정확값
$$P\Big(\max_{s\le1}X_s\ge1\Big)=1-\Phi\Big(\frac{1-0.5}{1}\Big)+e^{2\cdot0.5\cdot1}\Big(1-\Phi\Big(\frac{1+0.5}{1}\Big)\Big)$$
(06e 에서 유도) 와 비교하라. 어느 가정이 깨졌는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

참값 $=1-\Phi(0.5)+e^{1}\big(1-\Phi(1.5)\big)=0.30854+2.71828\times0.066807=0.490138$. 시뮬레이션은 $0.490\pm0.0035$ ($N=20000$) 를 주고, 순진한 반사값 $0.617$ 은 36 SE 밖이다.

깨진 것은 **대칭성**이다: 도달 후 $X$ 는 위로 갈 확률이 더 크므로 ($+0.5t$ 드리프트) 반사 경로의 확률이 원 경로와 다르다 — $P(T\le1,X_1<1)<P(T\le1,X_1>1)$. 강마르코프 성질과 경로 연속성은 그대로 성립한다 (드리프트 BM 도 연속이고 정지시간 이후 재출발한다). 06d 에서는 지수 마팅게일로, 06e 에서는 Girsanov 로 이 경우를 복구한다.

```python
S_drift = max_hit_stats(N_PATHS, 1000, 1.0, rng, level=1.0, drift=0.5)
p_drift_hat = mc_proportion(S_drift["M_exact"] >= 1)
p_drift_exact = 1 - norm.cdf(0.5) + np.exp(2 * 0.5 * 1.0) * (1 - norm.cdf(1.5))
p_naive = 2 * (1 - norm.cdf(0.5))
print(p_drift_hat, p_drift_exact, p_naive)   # ~0.490 ± 0.0035, 0.490138, 0.617075
print(f"naive reflection is {(p_naive - p_drift_hat.mean) / p_drift_hat.se:.0f} SE away")
assert_close(p_drift_hat, p_drift_exact, k=4, name="P(max X >= 1) drifted BM")
```

</details>

## 7. 정리

- 반사 원리: $a>0$ 에 대해 $P(M_t\ge a)=2P(B_t\ge a)$, $P(M_t\ge a,B_t\le b)=P(B_t\ge2a-b)$; $M_t\overset d=|B_t|$, $E[M_t]=\sqrt{2t/\pi}$. 근거는 강마르코프 + 대칭 + 연속.
- 도달시간 $T_a\overset d=a^2/Z^2$: 밀도 $\frac{a}{\sqrt{2\pi t^3}}e^{-a^2/2t}$, 중앙값 $2.198a^2$, 평균 $\infty$ (꼬리 $t^{-1/2}$) — 반드시 도달하지만 '언제' 의 평균은 없다.
- 격자 최댓값은 항상 과소, 격자 도달시간은 항상 과대: 편향 $\approx$ 장벽 이동 $0.5826\sqrt{\Delta t}$, $\sqrt{\Delta t}$ 로만 감소.
- 구간별 브라운 다리 최댓값 $m=\frac12\big(x+y+\sqrt{(y-x)^2-2\Delta t\ln U}\big)$ 을 뽑으면 러닝 최댓값·도달확률이 편향 없이 나온다 (도달시각은 $\Delta t$ 이내).
- 아크사인 법칙: 양수 체류 비율·마지막 영점·최댓값 위치는 U 자형 밀도 $1/(\pi\sqrt{x(1-x)})$, $P(\le1/4)=1/3$. 마지막 영점은 격자 부호변화로 추정하면 편향 (다리 교차 확률 $e^{-2xy/\Delta t}$ 로 보정).

**trap 카드.** Q: 격자 시뮬레이션에서 $\hat P(M_1\ge1)$ 이 참값보다 3 SE 낮게 나왔다. 난수 탓인가? → A: 아니다, 편향이다. 격자 최댓값은 항상 참 최댓값 이하이고 장벽을 $0.58\sqrt{\Delta t}$ 만큼 민 효과를 낸다: $n=1000$ 이면 $-0.009$, 즉 $N=20000$ 의 SE 0.0033 의 2.7 배. 다리 보정을 쓰거나 $\Delta t$ 를 100 배 줄여라.

**다음 노트북: 06d.** 최댓값·도달시간을 '반사' 로 풀었다면 06d 는 같은 문제를 마팅게일 + 선택정지로 푼다: 구간 탈출 확률 $a/(a+b)$, 평균 탈출시간 $ab$ (도박꾼 파산의 네 번째 풀이), 그리고 오늘 E3 에서 깨진 드리프트 경우를 지수 마팅게일로 복구한다. 오늘의 다리 보정은 탈출 시뮬레이션에도 그대로 쓰인다.

log/progress.md 한 줄 템플릿: `- 2026-XX-XX 06c 반사 원리·최댓값·도달시간 — 예측 6/6 중 __ 맞음; compare 15 행 중 __ ok (편향 시연 3 행 제외); 막힌 곳: __`